# Telco Customer Churn Model
Train four customer clusters and an L2-regularized logistic regression classifier.

In [ ]:
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

df = pd.read_excel("Telco_customer_churn.xlsx")


## Clean data and define target
Geographic fields, identifiers, duplicate/post-outcome churn fields, and the target are excluded from predictors.

In [ ]:
TARGET = "Churn Value"
DROP_COLUMNS = ["Country", "State", "City", "Zip Code", "Lat Long", "Latitude", "Longitude", "CustomerID", "Count", "Churn Label", "Churn Score", "Churn Reason", TARGET]
df = df.copy()
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")
X = df.drop(columns=[c for c in DROP_COLUMNS if c in df.columns])
y = df[TARGET].astype(int)
print("Rows and columns:", df.shape)
print("Predictor columns:", X.columns.tolist())
display(y.value_counts(normalize=True))


## Train/test split
The two requested CSV exports contain predictor columns only, without the target variable.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_train.to_csv("training_data.csv", index=False)
X_test.to_csv("test_data.csv", index=False)
print(X_train.shape, X_test.shape)


## Impute, scale, and one-hot encode
Fit preprocessing on training data only; unknown future categories are handled safely.

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
preprocessor = ColumnTransformer([("numeric", numeric_pipeline, numeric_features), ("categorical", categorical_pipeline, categorical_features)], remainder="drop")
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
print("Encoded feature count:", X_train_processed.shape[1])


## Cluster customers into four groups
K-Means is fitted on training customers only; test customers are assigned to learned clusters.

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
train_clusters = kmeans.fit_predict(X_train_processed)
test_clusters = kmeans.predict(X_test_processed)
display(pd.Series(train_clusters + 1, name="Cluster").value_counts().sort_index())


## Train L2 logistic regression
Cluster IDs are added as one-hot indicators to the encoded customer predictors.

In [ ]:
X_train_model = np.column_stack([X_train_processed, np.eye(4)[train_clusters]])
X_test_model = np.column_stack([X_test_processed, np.eye(4)[test_clusters]])
classifier = LogisticRegression(penalty="l2", C=1.0, solver="lbfgs", max_iter=2000, random_state=42)
classifier.fit(X_train_model, y_train)


## Evaluate on held-out test set

In [ ]:
y_pred = classifier.predict(X_test_model)
y_probability = classifier.predict_proba(X_test_model)[:, 1]
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 3))
print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 3))
print("F1:", round(f1_score(y_test, y_pred, zero_division=0), 3))
print("ROC AUC:", round(roc_auc_score(y_test, y_probability), 3))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))


## Save model.pkl
The artifact contains preprocessing, K-Means, logistic regression, and the expected feature schema.

In [ ]:
model_artifact = {"preprocessor": preprocessor, "kmeans": kmeans, "classifier": classifier, "feature_columns": X_train.columns.tolist(), "numeric_features": numeric_features, "categorical_features": categorical_features, "excluded_columns": DROP_COLUMNS, "target_column": TARGET, "cluster_count": 4, "threshold": 0.5}
with open("model.pkl", "wb") as f:
    pickle.dump(model_artifact, f)
print("Successfully saved model.pkl")


## Verify saved model

In [ ]:
with open("model.pkl", "rb") as f:
    loaded = pickle.load(f)
test_transformed = loaded["preprocessor"].transform(X_test)
test_clusters_loaded = loaded["kmeans"].predict(test_transformed)
test_matrix = np.column_stack([test_transformed, np.eye(4)[test_clusters_loaded]])
print("First five probabilities:", loaded["classifier"].predict_proba(test_matrix)[:5, 1])
print("First five clusters:", test_clusters_loaded[:5] + 1)
